# Visualize Effect

This notebook shows what the imposed effects look like on a sample 2d image

In [ ]:
import hglm 
import nibabel as nib
import numpy as np

# load experiment
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp = hglm.experiment.ExperimentImageOnly.from_search(folder=folder,
                                                      sbj_regex=r'[\d]{6}',
                                                      img_glob_dict={'FA': '*_FA.nii.gz'})

# set x value of each image according to its index
b, num_img, num_vox = exp.y.shape
exp = hglm.experiment.Experiment(x=np.arange(num_img).reshape(1, num_img), 
                                 contrast=np.array(True), 
                                 add_bias=True, 
                                 y=exp.y, 
                                 mask_idx=exp.mask_idx)

# constrain ourselves to a slice (easier to visualize)
mask = exp.mask_idx > -1
mid = tuple(_xyz // 2 for _xyz in mask.shape)
mask[..., :mid[2]] = False
mask[..., mid[2] + 1:] = False
exp_slice = exp.apply_mask(mask)
exp_slice.mask_idx = exp_slice.mask_idx[..., mid[2]]

In [ ]:
import matplotlib.pyplot as plt
from skimage import measure

def plot_slice(y, mask_idx, mask=None, **kwargs):
    # nans transparent
    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color=(0, 0, 0, 0)) 
    
    # build image
    img = np.full(mask_idx.shape, np.nan)
    img[mask_idx > -1] = y
    im = plt.imshow(img, cmap=cmap, **kwargs)
    
    ax = plt.gca()
    ax.grid(False)
    ax.tick_params(labelbottom=False, labelleft=False)
    ax.set_facecolor('none')
    
    if mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=3, color='red')
    return im
    
def plot_all(exp, mask=None, scatter=False, scatter_n=None, htitle=None, vminmax=None, num_img=3, num_img_scatter=3, **kwargs):
    fig, ax = plt.subplots(1, num_img + scatter)
    fig.set_size_inches((15, 3.3))
    if vminmax is None:
        vmin, vmax = exp.y[0, ...].min(), exp.y[0, ...].max()
    else:
        vmin, vmax = vminmax
    
    img_idx = np.linspace(0, exp.y.shape[1] - 1, num=num_img, dtype=int)
    for _img_idx, _ax in zip(img_idx, ax):
        plt.sca(_ax)
        im = plot_slice(y=exp.y[0, _img_idx, :], 
                        mask_idx=exp_slice.mask_idx,
                        vmin=vmin, vmax=vmax, mask=mask,
                        **kwargs)
        
        _ax.set_title(f'img{_img_idx} (x={exp_slice.x[1, _img_idx]:g})')
        
    if scatter:
        # adds scatter plot of features
        y = exp.y[0, :, :]
        if mask is not None:
            y = y[:, exp.mask_idx[mask]]
            
        # get representative subset (less overwhelming visuals)
        num_vox = y.shape[1]
        if scatter_n is not None and scatter_n < num_vox:
            idx = np.linspace(0, num_vox - 1, scatter_n).astype(int)
            y = np.vstack([np.sort(row)[idx] for row in y])
            
        # plot with last x feature (avoids bias term)
        _x = exp.x[-1, :]
        x = np.broadcast_to(_x[:, None], y.shape)
        y_hat = y.mean(axis=1) @ np.linalg.pinv(exp.x) @ exp.x
        
        # trim to num_img_scatter
        img_idx = np.linspace(0, exp.y.shape[1] - 1, num=num_img_scatter, dtype=int)
        x = x[img_idx, :]
        y = y[img_idx, :]
        y_hat = y_hat[img_idx]
        
        # plot
        plt.sca(ax[-1])
        plt.plot(x.mean(axis=1), y_hat, color='k', linewidth=.5)
        plt.scatter(x=x.ravel(), y=y.flatten(), marker='s', s=13, c=y.flatten(),
                    vmin=vmin, vmax=vmax, cmap=im.get_cmap(), edgecolors='black', linewidths=0.4)
        
        labels = [f'x={val:g}\nimg{val:g}' for idx, val in enumerate(_x[img_idx])]
        ax[-1].set_xticks(_x[img_idx])
        ax[-1].set_xticklabels(labels)
        ax[-1].set_ylim(vmin, vmax)

    ax[0].set_ylabel(htitle)
    
    fig.tight_layout()

In [ ]:
# sample extent
ext_sphere = hglm.effect.ExtenterSphere(radius=10)
mask = ext_sphere(y=exp_slice.y, mask_idx=exp_slice.mask_idx, seed=0)

# build effects
exp_tit_list = []
for pval in [.9, .5, .1]:
    _exp, effect = exp_slice.impose_effect(pval=pval, mask=mask, seed=0) 
    title = f'F-Ratio:{effect.f_ratio:.4f}\nEffect P-val: {pval:.2e}\nRough: {effect.rough:.3f}'
    exp_tit_list.append((_exp, title))

# plot
vmin = min(exp.y.min() for exp, _ in exp_tit_list)
vmax = min(exp.y.max() for exp, _ in exp_tit_list)
for _exp, title in exp_tit_list:
    plot_all(_exp, mask=mask, scatter=True, scatter_n=25, htitle=title, vminmax=(vmin, vmax))

In [ ]:
# sample extent
ext_sphere = hglm.effect.ExtenterSphere(radius=10)
mask = ext_sphere(y=exp_slice.y, mask_idx=exp_slice.mask_idx, seed=0)

# build effects
exp_tit_list = []
for rough in [1, .1, .01]:
    _exp, effect = exp_slice.impose_effect(pval=.1, rough=rough, mask=mask, seed=0) 
    title = f'F-Ratio:{effect.f_ratio:.2f}\nRough: {effect.rough:.3f}'
    exp_tit_list.append((_exp, title))

# plot
img_idx = (0, 49, 99)
vmin = min(exp.y[:, img_idx, :].min() for exp, _ in exp_tit_list)
vmax = min(exp.y[:, img_idx, :].max() for exp, _ in exp_tit_list)
for _exp, title in exp_tit_list:
    plot_all(_exp, mask=mask, scatter=True, scatter_n=25, htitle=title, vminmax=(vmin, vmax))